In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ z= xW + b $$

$$ \diamonds \diamonds \diamonds $$

$$ t = \tanh(z) $$
$$ p = 0.5t + 0.5 $$
$$ \frac {\partial p}{\partial z} = 0.5 (1 - (2p-1)^2) $$

$$ \diamonds \diamonds \diamonds $$

$$ L = -y \log(p) - (1-y) \log(1-p) $$
$$ \frac{\partial L}{\partial z} = \frac{\partial L}{\partial p} \frac{\partial p}{\partial z} = \Bigg( \frac{p-y}{p(1-p)} \Bigg) \Bigg( 0.5 (1 - (2p-1)^2) \Bigg) $$
$$ \frac{\partial L}{\partial x} = \frac{\partial L}{\partial p} \frac{\partial p}{\partial x} = \Bigg( \frac{p-y}{p(1-p)} 0.5 (1 - (2p-1)^2) \Bigg) \cdot W^T $$
$$ \frac{\partial L}{\partial W} = \frac{\partial L}{\partial p} \frac{\partial p}{\partial W} = x^T \cdot \Bigg( \frac{p-y}{p(1-p)} 0.5 (1 - (2p-1)^2) \Bigg) $$
$$ \frac{\partial L}{\partial b} = \frac{\partial L}{\partial p} \frac{\partial p}{\partial b} = \frac{p-y}{p(1-p)} 0.5 (1 - (2p-1)^2) $$

$$ \diamonds \diamonds \diamonds $$

In [ ]:
import torch as tr
import torch.nn as nn
import torch.optim as optim
import torch.autograd as autograd

import import_ipynb
import tanh # type: ignore
import linear # type: ignore
import binary_cross_entropy # type: ignore
from common import assert_eq, average_run, test_perceptron_boolean # type: ignore


class Per_Lin_Tanh_BCE_Gradient_Function(autograd.Function):
    @staticmethod
    def forward(ctx, x, W, b, y):
        z = linear.linear(x, W, b)
        t = tanh.tanh(z)
        p = 0.5 * t + 0.5
        L = binary_cross_entropy.binary_cross_entropy(p, y)

        ctx.save_for_backward(x, W, b, y, p)

        return L

    @staticmethod
    def backward(ctx, dF_dL):
        (x, W, b, y, p) = ctx.saved_tensors

        S = x.shape[0]  # Samples
        FI = x.shape[1] # Features In
        FO = W.shape[1] # Features Out / Neurons

        assert_eq(x.shape, (S, FI))
        assert_eq(W.shape, (FI, FO))
        assert_eq(b.shape, (FO,))
        assert_eq(y.shape, (S, FO))
        assert_eq(p.shape, (S, FO))

        dp_dz = 0.5 * (1 - (2 * p - 1) ** 2)
        assert_eq(dp_dz.shape, (S, FO))

        dL_dp = (p - y) / (p * (1 - p)) / S
        assert_eq(dL_dp.shape, (S, FO))

        dL_dz = dL_dp * dp_dz
        assert_eq(dL_dz.shape, (S, FO))

        # (S, FI) = (S, FO) @ (FI, FO).T
        dL_dx = dL_dz @ W.t()
        assert_eq(dL_dx.shape, (S, FI))
        
        # (FI, FO) = (S, FI).T @ (S, FO)
        dL_dW = x.t() @ dL_dz
        assert_eq(dL_dW.shape, (FI, FO))

        # `dL_dz` is already averaged over samples, 
        # so summing gives the correct gradient
        dL_db = dL_dz.sum(dim=0)
        assert_eq(dL_db.shape, (FO,))

        return (dF_dL * dL_dx, dF_dL * dL_dW, dF_dL * dL_db, None)
    

class Per_Lin_Tanh_BCE_Gradient(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        # 
        self.weight = nn.Parameter(tr.randn(out_features, in_features, dtype=tr.float32))
        self.bias = nn.Parameter(tr.randn(out_features, dtype=tr.float32))

    def forward(self, x, y):
        return Per_Lin_Tanh_BCE_Gradient_Function.apply(x, self.weight.T, self.bias, y)
    
    # Extension
    # To facilitate testing and experimentation various perceptrons.
    def learn(self, x, y):
        return self(x, y)

    # Extension
    # To facilitate testing and experimentation various perceptrons.
    def predict(self, x):
        with tr.no_grad():
            z = linear.linear(x, self.weight.T, self.bias)
            t = tanh.tanh(z)
            p = 0.5 * t + 0.5
            return p >= 0.5


def test_Per_Lin_Tanh_BCE_Gradient(bool_fn, epochs, lr=0.1):
    return test_perceptron_boolean(Per_Lin_Tanh_BCE_Gradient(2, 1), bool_fn, epochs=epochs, lr=lr)


if __name__ == "__main__":
    N=10

    logical_and = tr.logical_and
    print(f" AND, {5*N} epochs: {average_run(1*N, lambda: test_Per_Lin_Tanh_BCE_Gradient(logical_and, epochs=5*N)):.2f}")

    logical_or = tr.logical_or
    print(f"  OR, {5*N} epochs: {average_run(1*N, lambda: test_Per_Lin_Tanh_BCE_Gradient(logical_or, epochs=5*N)):.2f}")

    logical_nand = lambda x1, x2: tr.logical_not(tr.logical_and(x1, x2))
    print(f"NAND, {5*N} epochs: {average_run(1*N, lambda: test_Per_Lin_Tanh_BCE_Gradient(logical_nand, epochs=5*N)):.2f}")

    logical_xor = lambda x1, x2: tr.logical_xor(x1, x2)
    print(f" XOR, {5*N} epochs: {average_run(1*N, lambda: test_Per_Lin_Tanh_BCE_Gradient(logical_xor, epochs=5*N)):.2f}")


 AND, 500 epochs: 1.00
  OR, 500 epochs: 1.00
NAND, 500 epochs: 1.00
 XOR, 500 epochs: 0.49
